In [ ]:
import pandas as pd
import psycopg2
import DATABASE_CONFIG

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG.DB_NAME,
    user=DATABASE_CONFIG.DB_USER,
    password=DATABASE_CONFIG.DB_PASSWORD,
    host=DATABASE_CONFIG.DB_HOST,
    port=DATABASE_CONFIG.DB_PORT
)

cursor = conn.cursor()

In [7]:
df = pd.read_csv('../datasets/DesmatamentoUnidadeConservacao.csv', sep=';')

df = df.rename(columns={
    'year': 'ano',
    'area km²': 'area_km2',
    'consunit': 'unidade_conservacao',
})

print(df.columns)
df['area_km2'] = df['area_km2'].str.replace(',', '.')

# for _, row in df.iterrows():
#     print(row)


Index(['ano', 'area_km2', 'unidade_conservacao'], dtype='object')


In [8]:
# Armazena dados prontos para inserir depois
dados_desmatamento = []

for _, row in df.iterrows():
    ano = row['ano']
    area_km2 = row['area_km2']
    unidade_conservacao = row['unidade_conservacao']
    unidade_conservacao = unidade_conservacao.upper()

    if unidade_conservacao.startswith("APA") or unidade_conservacao.startswith("RPPN"):
        if unidade_conservacao.startswith("APA"):
            unidade_conservacao = unidade_conservacao.removeprefix("APA").strip()
        else:
            unidade_conservacao = unidade_conservacao.removeprefix("RPPN").strip()

        unidade_conservacao = unidade_conservacao.title()
    
        cursor.execute("""
            SELECT id_unidade_conservacao FROM unidade_conservacao
            WHERE nome ILIKE '%%' || %s
        """, (unidade_conservacao,))
    else:
        unidade_conservacao = unidade_conservacao.title()

        cursor.execute("""
            SELECT id_unidade_conservacao FROM unidade_conservacao
            WHERE nome = %s
        """, (unidade_conservacao,))
    
    result = cursor.fetchone()
    id_unidade_conservacao = result[0] if result else None

    if id_unidade_conservacao is None:
        continue
        
    dados_desmatamento.append((ano, area_km2, id_unidade_conservacao))

In [9]:
from psycopg2.extras import execute_values

# Agora, insere **em lote**:
query = """
    INSERT INTO relatorio_desmatamento
    (ano, area_km2, id_unidade_conservacao)
    VALUES %s
"""
execute_values(cursor, query, dados_desmatamento)

# Finaliza
conn.commit()

In [ ]:
conn.rollback()

In [10]:
cursor.close()
conn.close()